# Flipkart Gridlock 2.0 — Inference Demo

This notebook is a thin, annotated walkthrough of `scripts/predict.py`. It imports
directly from `src/gridlock` rather than redefining any logic inline, so it can never
drift out of sync with the actual pipeline the way this project's earlier
notebook/training-script pair did. See `docs/APPROACH.md` for the full methodology.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from xgboost import XGBRegressor

from gridlock import config
from gridlock.features import ALL_FEATURES, build_feature_pipeline, transform

print("Model path:", config.MODEL_PATH)

## 1. Load data and build features

`build_feature_pipeline` fits the day-48 historical profile and the road/weather
encoders once, from train (and test feature columns, never test labels).

In [ ]:
train_df = pd.read_csv(config.TRAIN_PATH)
test_df = pd.read_csv(config.TEST_PATH)

_, test_df, encoders = build_feature_pipeline(train_df, test_df)
test_df = transform(test_df, encoders)

test_df[ALL_FEATURES].head()

## 2. Load the trained model

Run `python scripts/train.py` first if `artifacts/spatial_model.json` doesn't exist yet.

In [ ]:
model = XGBRegressor()
model.load_model(config.MODEL_PATH)

## 3. Predict and write the submission file

Demand is bounded in `[0, 1]` by the competition's own formulation, so predictions are clipped.

In [ ]:
predictions = model.predict(test_df[ALL_FEATURES]).clip(0, 1)

submission = pd.DataFrame({"Index": test_df["Index"], "demand": predictions})
config.ARTIFACTS_DIR.mkdir(exist_ok=True)
submission.to_csv(config.SUBMISSION_PATH, index=False)

print(f"Saved {len(submission)} predictions to {config.SUBMISSION_PATH}")
submission.head()